# Detección y Modificación de Color de Cabello (Google Colab)
Este notebook entrena un modelo YOLOv8-seg para detectar cabello y modificar su color. Diseñado para ejecución en Google Colab con GPU.

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics opencv-python numpy matplotlib Pillow tqdm

In [ ]:
from pathlib import Path
import os

# Rutas del proyecto
BASE_DIR = Path('/content/drive/MyDrive/ActividadEvaluativa1_SantiagoArbelaez_FedericoAlvarez/ActividadEvaluativa3/hair_color_detection')
DATA_DIR = BASE_DIR / 'data'
IMG_DIR = DATA_DIR / 'CelebA-HQ-img'
MASK_DIR = DATA_DIR / 'CelebAMask-HQ-mask-anno'
YOLO_DIR = DATA_DIR / 'yolo_format'
RESULTS_DIR = BASE_DIR / 'results'
WEIGHTS_DIR = RESULTS_DIR / 'weights'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
RECOLORED_DIR = RESULTS_DIR / 'recolored'

paths_to_check = [BASE_DIR, DATA_DIR, IMG_DIR, MASK_DIR]
for p in paths_to_check:
    if p.exists():
        print(f"✅ Encontrado: {p}")
    else:
        print(f"❌ No encontrado: {p}")

# Crear directorios si no existen
for d in [YOLO_DIR, WEIGHTS_DIR, PREDICTIONS_DIR, RECOLORED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Rutas configuradas correctamente.")

USE_SUBSET = True
NUM_IMAGES = 5000

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Exploración del dataset
images = list(IMG_DIR.glob('*.jpg'))
print(f"Total imágenes en CelebA-HQ-img: {len(images)}")

masks = list(MASK_DIR.glob('*/*_hair.png'))
print(f"Total máscaras de cabello: {len(masks)}")

# Visualizar 5 pares
fig, axes = plt.subplots(5, 2, figsize=(10, 20))
for i in range(min(5, len(masks))):
    mask_path = masks[i]
    img_id = int(mask_path.name.split('_')[0])
    img_path = IMG_DIR / f"{img_id}.jpg"

    if img_path.exists():
        img = Image.open(img_path)
        mask = Image.open(mask_path)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Imagen {img_id}")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(img)
        axes[i, 1].imshow(mask, alpha=0.5, cmap='jet')
        axes[i, 1].set_title(f"Máscara {img_id}")
        axes[i, 1].axis('off')
plt.tight_layout()
plt.show()
print("✅ Celda completada")

⚠️ **Advertencia sobre conversión:**
La conversión de 30,000 imágenes puede tomar 20-40 minutos en Colab.
Se recomienda usar solo las primeras 5,000 para pruebas rápidas (USE_SUBSET=True).

In [ ]:
import cv2
import shutil
from tqdm import tqdm
import random

def convert_to_yolo():
    # Crear estructura YOLO
    for split in ['train', 'val']:
        (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

    masks = list(MASK_DIR.glob('*/*_hair.png'))
    random.seed(42)
    random.shuffle(masks)

    if USE_SUBSET:
        masks = masks[:NUM_IMAGES]
        print(f"Usando subset de {NUM_IMAGES} imágenes.")

    split_idx = int(len(masks) * 0.8)
    train_masks = masks[:split_idx]
    val_masks = masks[split_idx:]

    def process_split(split_masks, split_name):
        processed = 0
        for mask_path in tqdm(split_masks, desc=f"Procesando {split_name}"):
            img_id = int(mask_path.name.split('_')[0])
            label_path = YOLO_DIR / 'labels' / split_name / f"{img_id}.txt"
            
            # Checkpoint: reanudar si ya existe
            if label_path.exists():
                processed += 1
                continue

            img_path = IMG_DIR / f"{img_id}.jpg"
            if not img_path.exists():
                print(f"Warning: No se encontró imagen para {mask_path.name}")
                continue

            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None: continue
            h, w = mask.shape
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            with open(label_path, 'w') as f:
                for contour in contours:
                    if len(contour) < 3: continue
                    contour = contour.squeeze()
                    if contour.ndim == 1: contour = contour.reshape(1, 2)
                    points = []
                    for pt in contour:
                        points.append(f"{pt[0]/w:.6f} {pt[1]/h:.6f}")
                    f.write(f"0 {' '.join(points)}\n")

            dest_img = YOLO_DIR / 'images' / split_name / f"{img_id}.jpg"
            shutil.copy(img_path, dest_img)
            processed += 1
        return processed

    train_count = process_split(train_masks, 'train')
    val_count = process_split(val_masks, 'val')

    print(f"Total procesadas: {train_count + val_count}")
    print(f"Total train: {train_count}")
    print(f"Total val: {val_count}")

# Preguntar si reejecutar (simulado para script)
print("Verificando si la conversión ya fue completada...")
convert_to_yolo()
print("✅ Celda completada")

In [ ]:
import yaml

data_yaml_path = YOLO_DIR / 'data.yaml'
yaml_content = {
    'path': str(YOLO_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['hair']
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)
print(f"data.yaml creado en: {data_yaml_path}")
print("✅ Celda completada")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')
model.train(
    data=str(YOLO_DIR / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device='cuda',
    project=str(WEIGHTS_DIR),
    name='hair_seg',
    exist_ok=True
)
print("✅ Celda completada")

In [ ]:
best_pt = WEIGHTS_DIR / 'hair_seg' / 'weights' / 'best.pt'
if best_pt.exists():
    print(f"Modelo guardado: {best_pt}")
    print(f"Tamaño: {best_pt.stat().st_size / 1e6:.1f} MB")
else:
    print("El modelo no se encontró.")
print("✅ Celda completada")

In [ ]:
# Cargar el mejor modelo
best_model = YOLO(str(WEIGHTS_DIR / 'hair_seg' / 'weights' / 'best.pt'))

# Evaluar en el conjunto de validación
metrics = best_model.val()

print(f"mAP50: {metrics.seg.map50:.4f}")
print(f"mAP50-95: {metrics.seg.map:.4f}")
if len(metrics.seg.p) > 0:
    print(f"Precision: {metrics.seg.p[0]:.4f}")
    print(f"Recall: {metrics.seg.r[0]:.4f}")
print("✅ Celda completada")

In [ ]:
def infer_and_visualize(img_path):
    img_path = Path(img_path)
    if not img_path.exists():
        print(f"Error: Imagen no encontrada {img_path}")
        return

    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = best_model(str(img_path))
    result = results[0]

    if result.masks is None:
        print("No se detectó cabello en la imagen.")
        return

    mask_data = result.masks.data[0].cpu().numpy()
    mask_resized = cv2.resize(mask_data, (img.shape[1], img.shape[0]))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(mask_resized, cmap='gray')
    axes[1].set_title("Máscara Binaria")
    axes[1].axis('off')

    overlay = img_rgb.copy()
    overlay[mask_resized > 0.5] = [255, 0, 0]
    axes[2].imshow(cv2.addWeighted(img_rgb, 0.5, overlay, 0.5, 0))
    axes[2].set_title("Superposición")
    axes[2].axis('off')

    plt.show()

    save_path = PREDICTIONS_DIR / f"pred_{img_path.name}"
    fig.savefig(save_path)
    print(f"Predicción guardada en {save_path}")

test_images = list((YOLO_DIR / 'images' / 'val').glob('*.jpg'))
if test_images:
    infer_and_visualize(test_images[0])
print("✅ Celda completada")

In [ ]:
def change_hair_color(img_path, hue_shift, saturation_scale=1.2):
    img_path = Path(img_path)
    img = cv2.imread(str(img_path))
    if img is None: return None

    results = best_model(str(img_path), verbose=False)
    result = results[0]

    if result.masks is None:
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask_data = result.masks.data[0].cpu().numpy()
    mask_resized = cv2.resize(mask_data, (img.shape[1], img.shape[0]))
    binary_mask = (mask_resized > 0.5).astype(np.uint8) * 255

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hair_hsv = cv2.bitwise_and(hsv, hsv, mask=binary_mask)

    h, s, v = cv2.split(hair_hsv)
    mask_bool = binary_mask > 0
    h = h.astype(np.int16)
    h[mask_bool] = (h[mask_bool] + hue_shift) % 180
    h = h.astype(np.uint8)

    s = s.astype(np.float32)
    s[mask_bool] = np.clip(s[mask_bool] * saturation_scale, 0, 255)
    s = s.astype(np.uint8)

    modified_hair_hsv = cv2.merge([h, s, v])
    modified_hair_bgr = cv2.cvtColor(modified_hair_hsv, cv2.COLOR_HSV2BGR)

    img_no_hair = cv2.bitwise_and(img, img, mask=cv2.bitwise_not(binary_mask))
    final_bgr = cv2.add(img_no_hair, modified_hair_bgr)
    final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)

    save_path = RECOLORED_DIR / f"recolored_h{hue_shift}_{img_path.name}"
    cv2.imwrite(str(save_path), cv2.cvtColor(final_rgb, cv2.COLOR_RGB2BGR))

    return final_rgb
print("✅ Celda completada")

In [ ]:
# Demo Interactiva
demo_images = test_images[:3] if len(test_images) >= 3 else test_images

fig, axes = plt.subplots(len(demo_images), 4, figsize=(16, 4*len(demo_images)))

colors = {
    "Rojo (0)": 0,
    "Verde (60)": 60,
    "Azul (120)": 120
}

for i, img_path in enumerate(demo_images):
    original = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    axes[i, 0].imshow(original)
    axes[i, 0].set_title("Original")
    axes[i, 0].axis('off')

    col = 1
    for color_name, hue in colors.items():
        recolored = change_hair_color(img_path, hue)
        axes[i, col].imshow(recolored)
        axes[i, col].set_title(color_name)
        axes[i, col].axis('off')
        col += 1

plt.tight_layout()
plt.show()
print("✅ Celda completada")